### Transmon Qubit with Readout Resonator — Eigenmode Simulation

This notebook replicates the
[Palace `transmon` example](https://github.com/awslabs/palace/tree/main/examples/transmon)
in gdsfactory. That example defines a superconducting transmon qubit coupled to a
quarter-wave readout resonator using
[DeviceLayout.jl](https://aws-cqc.github.io/DeviceLayout.jl/). Here we port the
`SingleTransmon` geometry to gdsfactory (using the QPDK technology), build the
sapphire/vacuum layer stack, and run an eigenmode simulation with the gsim Palace
infrastructure.

The system consists of:

1. A **transmon qubit** — a rectangular capacitor island shunted by a Josephson
   junction (modelled as a lumped L/C element).
2. A **quarter-wave coplanar resonator** — a meandering CPW with a claw coupler
   to the transmon.
3. A **feedline** — a straight CPW for readout, terminated by two 50 Ω lumped
   ports.
4. **Airbridges** — metal staples that tie the ground planes together across the
   CPW.

[Palace](https://awslabs.github.io/palace/) is an open-source 3D electromagnetic
simulator supporting eigenmode, driven (S-parameter), and electrostatic
simulations.

**Requirements:**

- Quantum PDK: `uv pip install qpdk`
- [GDSFactory+](https://gdsfactory.com) account for cloud simulation


In [ ]:
import gdsfactory as gf
import klayout.db as kdb
import numpy as np
from qpdk import PDK
from qpdk.tech import LAYER, coplanar_waveguide

PDK.activate()

### 1. Geometry — `SingleTransmon` ported to gdsfactory

The parameters below are the defaults of
`SingleTransmon.single_transmon()` in DeviceLayout.jl. The construction follows
the Julia source closely:

- The readout resonator path is built segment by segment (straights, circular
  turns and a serpentine `meander`) and extruded with the QPDK CPW
  cross-section, which draws the etched gaps on `M1_ETCH`.
- The claw coupler and the transmon island cutout are polygons on `M1_ETCH`,
  while the junction leads are on `M1_DRAW`.
- The feedline is a straight CPW rotated to run along +x.
- Staple airbridges are placed at the path midpoints, matching
  `add_bridges!(..., spacing=300µm)`.

The metal is then reconstructed as `sim_area - M1_ETCH` (with `M1_DRAW` filled
back in), exactly like DeviceLayout's
`metal = (writeable_area - metal_negative) + metal_positive`.


In [ ]:
# ---------------------------------------------------------------------------
# DeviceLayout SingleTransmon parameters (transmon.jl defaults)
# ---------------------------------------------------------------------------
SHIELD_WIDTH = 2.0  # claw shield ground plane width
CLAW_GAP = 6.0  # claw capacitor gap
CLAW_TRACE = 34.0  # claw finger width
CLAW_LENGTH = 121.0  # claw finger length
CAP_WIDTH = 24.0  # transmon island width
CAP_LENGTH = 620.0  # transmon island length
CAP_GAP = 30.0  # gap around the island
TOTAL_LENGTH = 5000.0  # total electrical length of the resonator
N_MEANDER_TURNS = 5  # number of meander turns
HANGER_LENGTH = 500.0  # hanger length between coupler and meander
BEND_RADIUS = 50.0  # meander bend radius

CPW_WIDTH = 10.0  # CPW centre conductor width
CPW_GAP = 6.0  # CPW gap width
COUPLING_LENGTH = 400.0  # length of the feedline coupling section
COUPLING_GAP = 5.0  # gap between feedline and resonator ground planes
ARM_LENGTH_REF = 428.0  # arm length used to derive total_y_length

JUNCTION_GAP = 12.0  # gap on the junction side of the island
JUNCTION_WIDTH = 1.0  # junction (and lead) width
JUNCTION_LEAD_GAP = 1.0  # length of the lumped-element junction

EXTENT = CPW_GAP + CPW_WIDTH / 2  # SimpleCPW extent (half ground-plane width)
GRASP_WIDTH = CAP_WIDTH + 2 * CAP_GAP
CUTOUT_HEIGHT = JUNCTION_GAP + CAP_LENGTH + CAP_GAP
TOTAL_Y_LENGTH = (
    ARM_LENGTH_REF
    + COUPLING_GAP
    + EXTENT
    + HANGER_LENGTH
    + (3 + N_MEANDER_TURNS * 2) * BEND_RADIUS
)
READOUT_LENGTH = 2700.0  # feedline length (2700um lumped ports / 4mm wave ports)
TY = TOTAL_Y_LENGTH + CUTOUT_HEIGHT  # translation that fuses resonator to transmon

XS = coplanar_waveguide(width=CPW_WIDTH, gap=CPW_GAP)

In [ ]:
from itertools import cycle, islice
from math import pi

DBU = 0.001  # um per database unit (1 nm)


def rect(x0, y0, x1, y1):
    """Axis-aligned rectangle in um as a klayout DBox."""
    return kdb.DBox(min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1))


def rect_center(cx, cy, w, h):
    """Rectangle of size (w, h) centred at (cx, cy) in um."""
    return rect(cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)


def as_region(*boxes):
    """Build a klayout Region (in database units) from DBoxes given in um."""
    region = kdb.Region()
    for box in boxes:
        region.insert(box.to_itype(DBU))
    return region


def add_region(component, layer, region):
    """Insert a klayout Region onto a component layer."""
    component.kdb_cell.shapes(component.kcl.layer(*layer)).insert(region)


def path_length(points):
    seg = np.diff(points, axis=0)
    return float(np.sum(np.hypot(seg[:, 0], seg[:, 1])))


def resonator_segments():
    """DeviceLayout ExampleClawedMeanderReadout path construction."""
    n_bends = 3 + 2 * N_MEANDER_TURNS
    arm_length = (
        TOTAL_Y_LENGTH
        - HANGER_LENGTH
        - n_bends * BEND_RADIUS
        - COUPLING_GAP
        - CPW_GAP
        - CPW_WIDTH / 2
        - SHIELD_WIDTH
        - 2 * CLAW_GAP
        - CLAW_TRACE
    )
    straight_length = (
        TOTAL_LENGTH
        - 3 * COUPLING_LENGTH / 2
        - n_bends * pi * BEND_RADIUS / 2
        - arm_length
        - HANGER_LENGTH
    ) / N_MEANDER_TURNS

    segments = [
        ("straight", COUPLING_LENGTH),
        ("turn", -90.0),
        ("straight", HANGER_LENGTH),
        ("turn", -90.0),
        ("straight", straight_length / 2 + COUPLING_LENGTH / 2),
        ("turn", 180.0),
    ]

    # DeviceLayout `meander!`: alternate straights and opposite U-turns.
    alpha = -180.0
    unit = straight_length + BEND_RADIUS * abs(np.deg2rad(alpha))
    meander_length = (
        (N_MEANDER_TURNS - 1) * (straight_length + pi * BEND_RADIUS)
        + straight_length / 2
        - BEND_RADIUS
    )
    ratio = meander_length / unit
    full_turns = int(np.floor(ratio))
    remainder = (ratio - full_turns) * unit
    for sign in islice(cycle((1, -1)), full_turns):
        segments.append(("straight", straight_length))
        segments.append(("turn", sign * alpha))
    segments.append(("straight", remainder))
    segments.append(("turn", -90.0))
    segments.append(("straight", arm_length))
    return segments, arm_length


def resonator_path():
    """The readout resonator centreline as a gdsfactory Path."""
    segments, _ = resonator_segments()
    path = gf.Path()
    for kind, value in segments:
        if kind == "straight":
            path.append(gf.path.straight(length=value))
        else:
            path.append(gf.path.arc(radius=BEND_RADIUS, angle=value))
    path.move((-COUPLING_LENGTH / 2, -(COUPLING_GAP + CPW_GAP + CPW_WIDTH / 2)))
    return path


def resonator_claw(pt0):
    """Claw coupler etch region (DeviceLayout ExampleClawedMeanderReadout)."""
    arm_trace = CPW_WIDTH
    cg, ct, cl, sw, gw = CLAW_GAP, CLAW_TRACE, CLAW_LENGTH, SHIELD_WIDTH, GRASP_WIDTH
    px, py = pt0

    hole1 = rect_center(px - arm_trace / 2, py - cg, arm_trace, cg)
    top1 = hole1.top
    hole2 = rect_center(
        px - arm_trace / 2,
        top1 - (ct + 2 * cg) / 2,
        gw + 2 * sw + 4 * cg + 2 * ct,
        ct + 2 * cg,
    )
    hole3 = rect(
        hole2.left,
        hole2.bottom - (sw + cl + cg),
        hole2.left + (ct + 2 * cg),
        hole2.bottom,
    )
    hole4 = rect(hole2.right - (ct + 2 * cg), hole3.bottom, hole2.right, hole3.top)

    claw1 = rect_center(px - arm_trace / 2, py - cg, arm_trace, cg)
    claw2 = rect_center(
        px - arm_trace / 2,
        claw1.bottom - ct / 2,
        gw + 2 * sw + 2 * cg + 2 * ct,
        ct,
    )
    claw3 = rect(
        claw2.left, claw2.bottom - (cg + sw + cl), claw2.left + ct, claw2.bottom
    )
    claw4 = rect(claw2.right - ct, claw3.bottom, claw2.right, claw3.top)

    holes = as_region(hole1, hole2, hole3, hole4)
    claws = as_region(claw1, claw2, claw3, claw4)
    return holes - claws


def transmon_geometry():
    """Transmon island cutout, junction leads and lumped-element rectangle."""
    cutout_rect = rect(
        -CAP_WIDTH / 2 - CAP_GAP, 0.0, CAP_WIDTH / 2 + CAP_GAP, CUTOUT_HEIGHT
    )
    island_rect = rect(
        -CAP_WIDTH / 2, JUNCTION_GAP, CAP_WIDTH / 2, JUNCTION_GAP + CAP_LENGTH
    )
    cutout = as_region(cutout_rect) - as_region(island_rect)

    jj_rect = rect_center(0.0, JUNCTION_GAP / 2, JUNCTION_WIDTH, JUNCTION_LEAD_GAP)
    top_lead = rect(-JUNCTION_WIDTH / 2, jj_rect.top, JUNCTION_WIDTH / 2, JUNCTION_GAP)
    bot_lead = rect(-JUNCTION_WIDTH / 2, 0.0, JUNCTION_WIDTH / 2, jj_rect.bottom)
    leads = as_region(top_lead, bot_lead)
    lumped = as_region(jj_rect)
    return cutout, leads, lumped


def bridge_cell():
    """Staple airbridge: an elevated bar (AB_DRAW) with two feet (AB_VIA).

    The bar spans the CPW over the centre conductor; the feet sit on the ground
    planes and are extruded to meet the bar, forming a staple.
    """
    c = gf.Component()
    bridge_width, ground_to_ground = 10.0, 2 * EXTENT
    foot_length = 5.0
    half = ground_to_ground / 2
    c.add_polygon(
        [
            (-bridge_width / 2, -half),
            (bridge_width / 2, -half),
            (bridge_width / 2, half),
            (-bridge_width / 2, half),
        ],
        layer=LAYER.AB_DRAW,
    )
    for sign in (1, -1):
        y0 = sign * half
        y1 = sign * (half + foot_length)
        c.add_polygon(
            [
                (-bridge_width / 2, y0),
                (bridge_width / 2, y0),
                (bridge_width / 2, y1),
                (-bridge_width / 2, y1),
            ],
            layer=LAYER.AB_VIA,
        )
    return c


def place_bridge(component, bridge, x, y, angle):
    ref = component << bridge
    ref.rotate(angle)
    ref.move((x, y))
    return ref


def path_point(points, arclen):
    """Point and local direction at a given arc length along a polyline."""
    seg = np.diff(points, axis=0)
    dist = np.hypot(seg[:, 0], seg[:, 1])
    cumulative = np.concatenate([[0.0], np.cumsum(dist)])
    index = int(np.searchsorted(cumulative, arclen))
    index = min(max(index, 1), len(points) - 1)
    frac = (arclen - cumulative[index - 1]) / max(dist[index - 1], 1e-9)
    point = points[index - 1] + frac * seg[index - 1]
    angle = np.rad2deg(np.arctan2(seg[index - 1][1], seg[index - 1][0]))
    return point, angle

In [ ]:
@gf.cell
def single_transmon() -> gf.Component:
    """Transmon qubit with readout resonator and feedline (DeviceLayout port)."""
    c = gf.Component()

    # --- Readout resonator CPW path ---
    path = resonator_path()
    points = path.points
    resonator = c << path.extrude(XS)
    resonator.move((0.0, TY))

    # --- Claw coupler at the end of the resonator arm ---
    claw = resonator_claw(points[-1])
    add_region(c, LAYER.M1_ETCH, claw.moved(0, int(round(TY / DBU))))

    # --- Transmon island cutout, junction leads and lumped element ---
    cutout, leads, lumped = transmon_geometry()
    add_region(c, LAYER.M1_ETCH, cutout)
    add_region(c, LAYER.M1_DRAW, leads)
    add_region(c, LAYER.JJ_AREA, lumped)

    # --- Feedline (rotated to run along +x) ---
    feedline = c << gf.path.straight(length=READOUT_LENGTH).extrude(XS)
    feedline.move((-READOUT_LENGTH / 2, TY + EXTENT))

    # --- Airbridges: two on the resonator, two on the feedline ---
    bridge = bridge_cell()
    _, arm_length = resonator_segments()
    hanger_mid = COUPLING_LENGTH + BEND_RADIUS * pi / 2 + HANGER_LENGTH / 2
    for arclen in (hanger_mid, path_length(points) - arm_length / 2):
        point, angle = path_point(points, arclen)
        place_bridge(c, bridge, point[0], point[1] + TY, angle)
    for xlocal in (READOUT_LENGTH / 4, 3 * READOUT_LENGTH / 4):
        place_bridge(c, bridge, xlocal - READOUT_LENGTH / 2, TY + EXTENT, 0.0)

    # --- Ports: two feedline CPW ports and the junction lumped port ---
    c.add_port(
        name="o1",
        center=(-READOUT_LENGTH / 2 + CPW_WIDTH, TY + EXTENT),
        width=CPW_WIDTH,
        orientation=180,
        layer=LAYER.M1_DRAW,
    )
    c.add_port(
        name="o2",
        center=(READOUT_LENGTH / 2 - CPW_WIDTH, TY + EXTENT),
        width=CPW_WIDTH,
        orientation=0,
        layer=LAYER.M1_DRAW,
    )
    c.add_port(
        name="junction",
        center=(0.0, JUNCTION_GAP / 2),
        width=JUNCTION_WIDTH,
        orientation=90,
        layer=LAYER.JJ_AREA,
    )
    return c


component = single_transmon()
_c = component.copy()
_c.draw_ports()
_c

### 2. Convert etch layers to conductor geometry

Following the QPDK workflow, we draw a simulation area around the device,
subtract the etch (`M1_ETCH`) from it to obtain the conductor, and fill the
additive metal (`M1_DRAW`) back in. The substrate and vacuum layers share the
same simulation-area outline. The airbridge layers are carried through
separately.

The `sim_area` is centred on the device bounding box and sized like the Palace
example (`4 mm x 3.7 mm`).


In [ ]:
import warnings

from qpdk.tech import LAYER as QPDK_LAYER
from qpdk.utils import apply_additive_metals

from gsim.common.polygon_utils import decimate

SUBSTRATE_X, SUBSTRATE_Y = 4000.0, 3700.0

# gsim layer map (GDS layer, datatype)
SUBSTRATE_GDS = (1, 0)
SUPERCONDUCTOR_GDS = (2, 0)
VACUUM_GDS = (3, 0)
AIRBRIDGE_GDS = (10, 0)
AIRBRIDGE_VIA_GDS = (10, 1)
# The gsim mesh generator keys stack layers by GDS layer *number*, so the
# airbridge feet must not share layer number 10 with the bar. Remap them.
AIRBRIDGE_LEG_GDS = (12, 0)

# Add the simulation area and process additive metals.
device = component.copy()
bb = device.bbox()
area = gf.components.rectangle(
    size=(SUBSTRATE_X, SUBSTRATE_Y), layer=QPDK_LAYER.SIM_AREA, centered=True
)
area_ref = device << area
area_ref.move(((bb.left + bb.right) / 2, (bb.bottom + bb.top) / 2))

processed = apply_additive_metals(device.copy())

# Build the conductor component expected by gsim.
sim_area_layer = (QPDK_LAYER.SIM_AREA[0], QPDK_LAYER.SIM_AREA[1])
etch_layer = (QPDK_LAYER.M1_ETCH[0], QPDK_LAYER.M1_ETCH[1])

layout = processed.kdb_cell.layout()
sim_region = kdb.Region(
    processed.kdb_cell.begin_shapes_rec(layout.layer(*sim_area_layer))
)
etch_region = kdb.Region(processed.kdb_cell.begin_shapes_rec(layout.layer(*etch_layer)))

etch_polys = decimate(list(etch_region.each()))
etch_region = kdb.Region()
for poly in etch_polys:
    etch_region.insert(poly)

if sim_region.is_empty():
    warnings.warn("No polygons found on SIM_AREA", stacklevel=2)
if etch_region.is_empty():
    warnings.warn("No polygons found on M1_ETCH", stacklevel=2)

conductor_region = sim_region - etch_region

etched = gf.Component("etched_component")
el = etched.kdb_cell.layout()
for gds_layer, region in [
    (SUPERCONDUCTOR_GDS, conductor_region),
    (SUBSTRATE_GDS, sim_region),
    (VACUUM_GDS, sim_region),
    (
        AIRBRIDGE_GDS,
        kdb.Region(processed.kdb_cell.begin_shapes_rec(layout.layer(*AIRBRIDGE_GDS))),
    ),
    (
        AIRBRIDGE_LEG_GDS,
        kdb.Region(
            processed.kdb_cell.begin_shapes_rec(layout.layer(*AIRBRIDGE_VIA_GDS))
        ),
    ),
]:
    etched.kdb_cell.shapes(el.layer(*gds_layer)).insert(region)

for port in processed.ports:
    etched.add_port(name=port.name, port=port)

etched

### 3. Layer stack

The Palace example uses a 525 µm sapphire substrate (anisotropic permittivity
`(9.3, 9.3, 11.5)` with material axes rotated 37°) and vacuum above the metal.
The aluminium film is modelled as a perfect conductor (zero thickness); the
airbridge is an elevated 10 µm-tall conductor tied to ground by its feet.


In [ ]:
from gsim.common.stack import Layer, LayerStack
from gsim.common.stack.materials import MATERIALS_DB

SUBSTRATE_THICKNESS = 525.0
VIA_THICKNESS = 5.0
BRIDGE_THICKNESS = 5.0
VACUUM_THICKNESS = 1000.0

stack = LayerStack(pdk_name="qpdk")
stack.layers["SUBSTRATE"] = Layer(
    name="SUBSTRATE",
    gds_layer=SUBSTRATE_GDS,
    zmin=-SUBSTRATE_THICKNESS,
    zmax=0.0,
    thickness=SUBSTRATE_THICKNESS,
    material="sapphire",
    layer_type="substrate",
)
stack.layers["SUPERCONDUCTOR"] = Layer(
    name="SUPERCONDUCTOR",
    gds_layer=SUPERCONDUCTOR_GDS,
    zmin=0.0,
    zmax=0.0,
    thickness=0.0,
    material="aluminum",
    layer_type="conductor",
)
# Airbridge: feet from the ground plane up to the bar, then the elevated bar.
stack.layers["AIRBRIDGE_LEG"] = Layer(
    name="AIRBRIDGE_LEG",
    gds_layer=AIRBRIDGE_LEG_GDS,
    zmin=0.0,
    zmax=VIA_THICKNESS,
    thickness=VIA_THICKNESS,
    material="aluminum",
    layer_type="conductor",
)
stack.layers["AIRBRIDGE"] = Layer(
    name="AIRBRIDGE",
    gds_layer=AIRBRIDGE_GDS,
    zmin=VIA_THICKNESS,
    zmax=VIA_THICKNESS + BRIDGE_THICKNESS,
    thickness=BRIDGE_THICKNESS,
    material="aluminum",
    layer_type="conductor",
)
stack.layers["VACUUM"] = Layer(
    name="VACUUM",
    gds_layer=VACUUM_GDS,
    zmin=0.0,
    zmax=VACUUM_THICKNESS,
    thickness=VACUUM_THICKNESS,
    material="vacuum",
    layer_type="dielectric",
)
stack.dielectrics = [
    {
        "name": "substrate",
        "zmin": -SUBSTRATE_THICKNESS,
        "zmax": 0.0,
        "material": "sapphire",
    },
    {"name": "vacuum", "zmin": 0.0, "zmax": VACUUM_THICKNESS, "material": "vacuum"},
]

# Anisotropic sapphire, matching the Palace transmon example.
sapphire = MATERIALS_DB["sapphire"].to_dict()
sapphire.update(
    {
        "permittivity": [9.3, 9.3, 11.5],
        "permeability": [0.99999975, 0.99999975, 0.99999979],
        "loss_tangent": [3.0e-5, 3.0e-5, 8.6e-5],
        "material_axes": [[0.8, 0.6, 0.0], [-0.6, 0.8, 0.0], [0.0, 0.0, 1.0]],
    }
)
stack.materials = {
    "sapphire": sapphire,
    "aluminum": MATERIALS_DB["aluminum"].to_dict(),
    "vacuum": MATERIALS_DB["vacuum"].to_dict(),
}

### 4. Eigenmode simulation

Three lumped ports define the boundary conditions, matching the Palace config:

- `o1`, `o2` — 50 Ω resistive terminations at the feedline ends.
- `junction` — the Josephson junction, modelled as a lumped inductance
  `L = 14.86 nH` in parallel with a capacitance `C = 5.5 fF`.

The airbridge feet and elevated bar are extruded as finite-conductivity
aluminium volumes. We solve for the two lowest modes with a target of 4 GHz.


In [ ]:
from gsim.palace import EigenmodeSim

sim = EigenmodeSim()
sim.set_geometry(etched)
sim.set_stack(stack)

# Josephson junction as a lumped L/C element (no parallel resistance).
sim.add_port(
    "junction",
    layer="SUPERCONDUCTOR",
    length=JUNCTION_LEAD_GAP,
    inductance=14.860e-9,
    capacitance=5.5e-15,
    offset=0.0,
)

# 50 Ohm feedline terminations.
sim.add_cpw_port(
    "o1", layer="SUPERCONDUCTOR", s_width=CPW_WIDTH, gap_width=CPW_GAP, length=10.0
)
sim.add_cpw_port(
    "o2", layer="SUPERCONDUCTOR", s_width=CPW_WIDTH, gap_width=CPW_GAP, length=10.0
)

sim.set_eigenmode(num_modes=2, target=4e9, tolerance=1e-8, save=2)
sim.set_numerical(order=2, tolerance=1e-12, max_iterations=500)
sim.absorbing_boundary = True

In [ ]:
sim.set_output_dir("./sim_palace_transmon")
sim.mesh(preset="default", margin=0)
sim.plot_mesh(show_groups=["SUPERCONDUCTOR", "sapphire"])

In [ ]:
print(sim.validate_mesh())
config_path = sim.write_config()
print(config_path)

In [ ]:
results = sim.run(check_cache=True)
results

### Results

The eigenmode solver returns the two lowest modes. For the Palace reference
example (second order, coarse mesh) these are approximately:

| Mode | Frequency | Q |
|------|-----------|---|
| 1 (transmon) | 4.10 GHz | 1.9e4 |
| 2 (resonator) | 5.60 GHz | 7.9e3 |


In [ ]:
import csv

eig_path = results["eig.csv"]
with open(eig_path) as f:
    reader = csv.reader(f)
    header = next(reader)
    rows = [row for row in reader if row]

print(", ".join(h.strip() for h in header))
for row in rows:
    print(", ".join(cell.strip() for cell in row))